# Entrenamiento de modelos YOLO para detección de defectos en PCB

Este notebook implementa el entrenamiento de modelos YOLO para la detección automática de defectos en placas de circuito impreso.

Se comparan dos variantes de la misma familia de modelos:

- **YOLO11 Nano (`yolo11n`)**: variante de menor tamaño y costo computacional.
- **YOLO11 Small (`yolo11s`)**: variante de mayor capacidad representacional y costo computacional.

Ambos modelos se entrenan mediante *fine-tuning* a partir de pesos preentrenados, utilizando las mismas particiones de entrenamiento y validación y una configuración experimental comparable.

## Objetivos

- Realizar *fine-tuning* de YOLO11 Nano sobre DeepPCB.
- Realizar *fine-tuning* de YOLO11 Small sobre DeepPCB.
- Mantener condiciones experimentales comparables entre ambos modelos.
- Registrar métricas y artefactos de entrenamiento.
- Analizar la convergencia de los modelos.
- Comparar desempeño y costo computacional.
- Reservar la partición `test` exclusivamente para la evaluación final.

## 1. Importación de librerías y configuración

Se importan las dependencias necesarias y se definen las rutas del dataset procesado y de los resultados experimentales.

La partición `test` no será utilizada durante el entrenamiento ni para la selección de modelos.

In [1]:
from pathlib import Path
import sys

import torch
import ultralytics
from ultralytics import YOLO

In [2]:
# Detectar raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Dataset procesado
YOLO_DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_pcb_yolo"
)

DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"

# Directorio de resultados
RUNS_DIR = PROJECT_ROOT / "runs"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Dataset YOLO: {YOLO_DATASET_DIR}")
print(f"Configuración: {DATA_YAML_PATH}")
print(f"Resultados: {RUNS_DIR}")

Raíz del proyecto: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo
Dataset YOLO: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\data\processed\deep_pcb_yolo
Configuración: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\data\processed\deep_pcb_yolo\data.yaml
Resultados: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\runs


In [3]:
assert YOLO_DATASET_DIR.exists(), (
    "No se encontró el dataset procesado."
)

assert DATA_YAML_PATH.exists(), (
    "No se encontró data.yaml."
)

print("Rutas validadas correctamente.")

Rutas validadas correctamente.


## 2. Verificación del entorno de ejecución

Se verifica la disponibilidad de aceleración por GPU y las versiones de las principales librerías utilizadas.

Esta información resulta relevante para documentar la reproducibilidad de los experimentos y analizar posteriormente el costo computacional de cada variante del modelo.

In [4]:
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

print(f"\nCUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Versión CUDA: {torch.version.cuda}")
else:
    print("Dispositivo de entrenamiento disponible: CPU")

Python: 3.11.15
PyTorch: 2.12.1+cpu
Ultralytics: 8.4.87

CUDA disponible: False
Dispositivo de entrenamiento disponible: CPU


## 3. Corrida de prueba con YOLO11 Nano

Antes de ejecutar los entrenamientos finales, se realiza una corrida mínima de prueba utilizando YOLO11 Nano.

El objetivo de esta etapa no es obtener el mejor desempeño posible, sino validar que:

- el archivo `data.yaml` es leído correctamente;
- las imágenes y etiquetas se encuentran en la estructura esperada;
- las clases son interpretadas correctamente;
- el entrenamiento puede ejecutarse sin errores;
- se generan métricas y artefactos de salida.

Esta corrida se realiza en CPU y con pocas épocas, por lo que sus resultados no serán utilizados como desempeño final del modelo.

In [5]:
# Cargar modelo YOLO11 Nano preentrenado
model_debug = YOLO("yolo11n.pt")

# Corrida mínima de prueba en CPU
debug_results = model_debug.train(
    data=str(DATA_YAML_PATH),
    epochs=3,
    imgsz=640,
    batch=4,
    device="cpu",
    project=str(RUNS_DIR),
    name="yolo11n_debug",
    workers=0,
    seed=42,
    exist_ok=True
)

Ultralytics 8.4.87  Python-3.11.15 torch-2.12.1+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\data\processed\deep_pcb_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_debug, 

### 3.1 Resultados de la corrida de prueba

La corrida de prueba permitió validar el pipeline completo de entrenamiento en entorno local. Dado que el equipo no dispone de GPU compatible con CUDA, esta ejecución se realizó sobre CPU y con una cantidad reducida de épocas.

Los resultados obtenidos en esta etapa no se consideran representativos del desempeño final del modelo, sino que se utilizan únicamente como validación técnica previa al entrenamiento definitivo.

In [6]:
import pandas as pd

debug_run_dir = RUNS_DIR / "yolo11n_debug"
debug_results_path = debug_run_dir / "results.csv"

print(f"Carpeta debug: {debug_run_dir}")
print(f"Existe results.csv: {debug_results_path.exists()}")

if debug_results_path.exists():
    df_debug_results = pd.read_csv(debug_results_path)
    display(df_debug_results.tail())

Carpeta debug: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\runs\yolo11n_debug
Existe results.csv: True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,386.521,2.22885,3.79499,1.41162,0.27238,0.44231,0.27645,0.14502,1.60530,2.63955,1.20456,0.000332,0.000332,0.000332
1,2,790.970,1.61940,2.36138,1.14396,0.50907,0.56684,0.52416,0.26313,1.39340,2.03040,1.09689,0.000446,0.000446,0.000446
2,3,1167.490,1.50838,1.96502,1.09182,0.61395,0.66092,0.67171,0.37984,1.28857,1.72389,1.06002,0.000339,0.000339,0.000339


In [7]:
if debug_run_dir.exists():
    print("Archivos generados en la corrida debug:")
    for path in sorted(debug_run_dir.iterdir()):
        print(path.name)

Archivos generados en la corrida debug:
args.yaml
BoxF1_curve.png
BoxP_curve.png
BoxPR_curve.png
BoxR_curve.png
confusion_matrix.png
confusion_matrix_normalized.png
labels.jpg
results.csv
results.png
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
weights


Debido a que el entorno local no cuenta con GPU disponible, la corrida local se utilizó únicamente como validación técnica del pipeline. Los entrenamientos finales se ejecutarán en un entorno con GPU para reducir los tiempos de cómputo y permitir una comparación más adecuada entre YOLO11 Nano y YOLO11 Small.